In [ ]:
eval "$(~/miniconda3/bin/conda shell.bash hook)"
conda activate TACCO

# TACCO

In [ ]:
import os
os.chdir(path_to_wd)
import sys
sys.path.append(os.path.abspath('./Reproducibility/Scripts/Source/utility_functions'))
from scRNAseq_Visualization_Functions import *
import pickle
import anndata as ad
import scanpy as sc
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import scipy
import scipy.sparse as sp
from scipy import sparse
import tacco as tc
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['font.family']= 'Dejavu serif'
plt.rcParams['pdf.fonttype'] = 'truetype'
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
## Plotting options
highres = True
default_dpi = 200.0
if highres:
    matplotlib.rcParams['figure.dpi'] = 648.0
    hr_ext = '_hd'
else:
    matplotlib.rcParams['figure.dpi'] = default_dpi
    hr_ext = ''

axsize = np.array([4,3])*0.5

region_colors = tc.pl.get_default_colors([f'region_{i}' for i in range(4)], offset=17)
split_names = np.array([f'sub_{i}' for i in range(4)])
split_colors = tc.pl.get_default_colors(split_names, offset=12)

from matplotlib import cm, colors
vega_20 = list(map(colors.to_hex, cm.tab20.colors))

## Data loading

In [ ]:
## Data preprocessing
DOGMA = sc.read("Reproducibility/Data/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

adata_RNA  = DOGMA[:, DOGMA.var["modality"] == "Gene Expression"].copy()
adata_RNA = adata_RNA[adata_RNA.obs['RNA_HQ']=='HQ']
keep_mask = adata_RNA.var["black_list"].astype(str).str.lower().eq("keep")
adata_RNA = adata_RNA[:, keep_mask].copy()
adata_RNA.obs['celltype'] = adata_RNA.obs['celltype'].replace({'LUM': 'LUM', 'Normal': 'LUM'})

#####################################
ref = adata_RNA[adata_RNA.obs['celltype'].isin(['NE','MES','SQM'])==False].copy()

ref_Ta = ref[ref.obs["STAGE"].isin(["Early_L"])==True].copy()
ref_T1is = ref[ref.obs["STAGE"].isin(["Early_H"])==True].copy()
ref_T2 = ref[ref.obs["STAGE"].isin(["Advanced"])==True].copy()
ref_BCG = ref[(ref.obs["STAGE"].isin(['post_BCG'])) | (ref.obs['sample'] == 'BC_012')].copy() 

## Annotation

In [ ]:
def load_and_preprocess(data_dir, tmp_puck, division_factor=0.65):
    path = f"{data_dir}{tmp_puck}/{tmp_puck}_anndata.h5ad" 
    slideseq = sc.read_h5ad(path)
    coords_df = pd.read_csv(f"{data_dir}/{tmp_puck}/{tmp_puck}_MatchedBeadLocation.csv",index_col=0)
    coords_df = coords_df.loc[slideseq.obs_names,:]
    # from Pixels to Micrometers
    slideseq.obs["x"] = coords_df.loc[:, 'SPATIAL_1'] * division_factor
    slideseq.obs["y"] = coords_df.loc[:, 'SPATIAL_2'] * division_factor
    slideseq.obs["x"] = slideseq.obs["x"] - slideseq.obs["x"].min()
    slideseq.obs["y"] = slideseq.obs["y"] - slideseq.obs["y"].min()
    slideseq.X = slideseq.X.tocsc()
    slideseq.obs['sample'] = tmp_puck
    slideseq.obs['sample'] = slideseq.obs['sample'].astype('category')
    puck_all = slideseq[:, slideseq.X.sum(axis=0) != 0].copy()
    puck_all.obs['all_beads'] = 'all_beads'
    puck = puck_all[puck_all.X.sum(axis=1) >= 50].copy()
    return puck, puck_all

In [ ]:
def annotate_and_plot(puck, puck_all, reference, tmp_puck, fig_dir, save_dir):
    tc.tl.annotate(adata=puck, reference=reference_mapping[tmp_puck], method='OT', bisections=4, bisection_divisor=3, 
                   annotation_key='celltype', 
                   result_key='predictions', 
                   epsilon=0.005, lamb=0.001, multi_center=50)
    axsize = np.array([4,3])*0.5
    
    custom_order = ['CD4_Tn','CD4_Tcm','CD4_Tsen','CD4_T_CD26','CD4_CTL','CD4_Th17','CD4_Tfh-like','CD4_Tph-like','CD4_T_proliferative', 
                    'Treg_naive','Treg_effector', 'CD8_Tn', 'CD8_Tcm', 'CD8_Tem', 'CD8_Temra','CD8_Trm', 'CD8_Tex_1', 'CD8_Tex_2','CD8_T_proliferative', 
                    'NK_CD56_CD49a_Hi_CD103_Hi','NK_CD56_CD49a_Hi_CD103_Lo','NK_CD56_CD49a_Lo','NK_CD56_dim','ILC3', 'MAIT', 
                    'B_naive','B_memory','Atypical_B','GC_B','Plasma',
                    'Mono','MDSC-like','TAM_TREM2','TAM_FOLR2','cDC1','cDC2','mregDC','preDC','pDC', 'Mast',
                    'proCAF','iCAF_CD321','iCAF_SLC14A1','matCAF','matCAP','contCAP','vSMC', 
                    "Endothelial", "LUM"]
    
    # Reorder the cell type predictions in the puck.obsm['predictions'] dataframe
    puck.obsm['predictions'] = puck.obsm['predictions'].reindex(columns=custom_order)
    lineage_definitions = {
    'CD4_Tconv': ['CD4_Tn','CD4_Tcm','CD4_Tsen','CD4_T_CD26','CD4_CTL','CD4_Th17','CD4_Tfh-like','CD4_Tph-like','CD4_T_proliferative'],
    'Treg': ['Treg_naive','Treg_effector'], 
    'CD8_T':['CD8_Tn', 'CD8_Tcm', 'CD8_Tem', 'CD8_Temra', 'CD8_Trm', 'CD8_Tex_1', 'CD8_Tex_2','CD8_T_proliferative'],
    'NK_ILC':['NK_CD56_CD49a_Hi_CD103_Hi','NK_CD56_CD49a_Hi_CD103_Lo','NK_CD56_CD49a_Lo','NK_CD56_dim','ILC3', 'MAIT'],
    'MoMac' :['Mono','MDSC-like','TAM_FOLR2','TAM_TREM2'],
    'DC':['cDC1','cDC2','mregDC','preDC','pDC'],
    'Mast':['Mast'],
    'B': ['B_naive','B_memory','Atypical_B','GC_B'],
    'Plasma':['Plasma'],
    'MSC': ['proCAF','iCAF_CD321','iCAF_SLC14A1','matCAF','matCAP','contCAP','vSMC'],
    'Endothelial': ["Endothelial"],
    'Epithelial': ["LUM"]
    }

    new_df = pd.DataFrame(index=puck.obsm["predictions"].index)
    for new_col, original_cols in lineage_definitions.items():
        new_df[new_col] = puck.obsm["predictions"][original_cols].sum(axis=1)

    lineage = ['CD4_Tconv','Treg','CD8_T','NK_ILC','MoMac','DC','Mast','B','Plasma','MSC','Endothelial','Epithelial']

    lineage_colors = {'CD4_Tconv': '#023eff', 'Treg':'#ffc400' , 'CD8_T': '#00d7ff', 'NK_ILC': '#8b2be2', 
                      'MoMac': '#1ac938', 'DC': '#55a868', 'Mast': '#a3a3a3', 'B': '#e8000b', 'Plasma': '#ff7c00', 
                      'MSC':'#3c3c3c' , 'Endothelial': '#dd8452', 'Epithelial': '#9f4800'}
    
    puck.obsm["predictions_lineage"] = new_df

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    
    # Scatter plot: Cell type predictions
    full_tdatas = { sample: puck_all[df.index] for sample, df in puck_all.obs.groupby('sample') }
    sc1 = tc.pl.scatter(full_tdatas,'all_beads',colors={'all_beads':'#f5f5f5'},joint=True,point_size=3, ax=axs[0, 0]);
    tdatas = { sample: puck[df.index] for sample, df in puck.obs.groupby('sample') }
    sc1 = tc.pl.scatter(tdatas,keys='predictions', position_key=['x','y'], joint=True, point_size=3, axsize=axsize, 
                  noticks=False, axes_labels=['X','Y'], legend=True, ax=axs[0, 0])
    axs[0, 0].set_title('Cell Type Scatter Plot')

    # Composition plot: Cell type predictions
    tc.pl.compositions(puck, value_key='predictions', group_key='sample', legend=True, axsize=(2.4, 2.5), ax=axs[0, 1])
    axs[0, 1].set_title('Cell Type Composition Plot')    
    
    # Scatter plot: Lineage predictions
    full_tdatas = { sample: puck_all[df.index] for sample, df in puck_all.obs.groupby('sample') }
    sc2 = tc.pl.scatter(full_tdatas,'all_beads',colors={'all_beads':'#f5f5f5'},joint=True,point_size=3, ax=axs[1, 0]);
    tdatas = { sample: puck[df.index] for sample, df in puck.obs.groupby('sample') }
    sc2 = tc.pl.scatter(tdatas,colors=lineage_colors, keys='predictions_lineage', position_key=['x','y'], joint=True, point_size=3, axsize=axsize, 
                  noticks=False, axes_labels=['X','Y'], legend=True, ax=axs[1, 0])
    axs[1, 0].set_title('Lineage Scatter Plot')
    
    # Composition plot: Lineage predictions
    tc.pl.compositions(puck, colors=lineage_colors, value_key='predictions_lineage', group_key='sample', legend=True, axsize=(2.4, 2.5), ax=axs[1, 1])
    axs[1, 1].set_title('Lineage Composition Plot')

    # Adjust the legends to be outside the plot frames
    axs[0, 0].legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=2)
    axs[0, 1].legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=2)
    axs[1, 0].legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=2)
    axs[1, 1].legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=2)
    plt.tight_layout()
    plt.savefig(f"{fig_dir}combined_spatial_scatter_composition_plots_{tmp_puck}.pdf", bbox_inches='tight')
    plt.close()

    #########################
    # Save
    tmp_df = pd.DataFrame(puck.uns[f"predictions_mc{multi_center}"]).copy()
    tmp_df.to_pickle(f"{save_dir}puck_uns_predictions_mc{multi_center}_{tmp_puck}.pkl")
    del puck.uns[f"predictions_mc{multi_center}"]
    puck.obsm['predictions'].to_csv(f"{save_dir}puck_adata_mc{multi_center}_{tmp_puck}_celltype_composition_df.txt",sep ='\t')
    puck.write(f"{save_dir}puck_adata_mc{multi_center}_{tmp_puck}.h5ad")
    del puck

In [ ]:
reference_mapping = {
    'BC066': ref_T2,
    'BC071': ref_T2,
    'BC073': ref_T1is,
    'BC084': ref_Ta,
    'BC093': ref_Ta,
    'BC094': ref_T1is,
    'BC096': ref_BCG,
    'BC098': ref_BCG,
    'BC101': ref_Ta,
    'BC102': ref_Ta
}

In [ ]:
fig_dir = "Reproducibility/Results/Plots/Slide-seqV2/"
os.makedirs(fig_dir, exist_ok = True)

data_dir = "Reproducibility/Data/Slide-seqV2/"
os.makedirs(data_dir, exist_ok = True)

save_dir = "Reproducibility/Results/Slide-seqV2/"
os.makedirs(save_dir, exist_ok = True)

Spatial and composition bar plot

In [ ]:
# Fig.S11B and C
multi_center=50
puck_ids = ['BC084','BC093','BC101','BC102','BC094','BC073','BC071','BC066','BC096','BC098']
for tmp_puck in puck_ids:
    print(f'{tmp_puck}_start!')
    puck, puck_all = load_and_preprocess(data_dir, tmp_puck, division_factor=0.65)
    puck = annotate_and_plot(puck, puck_all, reference_mapping[tmp_puck], tmp_puck, fig_dir, save_dir)

QC spatial plot

In [ ]:
# utility function for plotting
def state_plot_grid(padding=None):
    # fixed mapping of sample to axis to sort by state
    fig,axs = tc.pl.subplots(4,3,x_padding=padding,y_padding=padding)
    state_axes = np.empty_like(axs, shape=(1,10))
    state_axes[0,0] = axs[0,0]
    state_axes[0,1] = axs[0,1]
    state_axes[0,2] = axs[0,2]
    state_axes[0,3] = axs[0,3]
    state_axes[0,4] = axs[1,0]
    state_axes[0,5] = axs[1,1]
    state_axes[0,6] = axs[1,2]
    state_axes[0,7] = axs[1,3]
    state_axes[0,8] = axs[2,0]
    state_axes[0,9] = axs[2,1]
    return state_axes[:, :10]

In [ ]:
# Fig.S11A
pucks_list = []

for i, puck_id in enumerate(puck_ids):
     file_path = f"{save_dir}puck_adata_mc{multi_center}_{puck_id}.h5ad"
     puck = sc.read(file_path)
     pucks_list.append(puck)

pucks = sc.concat(pucks_list, label='puck', index_unique='-')

cat_type = CategoricalDtype(categories=puck_ids, ordered=True)
pucks.obs['sample'] = pucks.obs['sample'].astype(cat_type)
pucks.obs['$log_{10}(UMI)$'] = np.log10(tc.sum(pucks.X,axis=1))

In [ ]:
tdatas = { sample: pucks[df.index] for sample, df in pucks.obs.groupby('sample') }
axs = state_plot_grid(padding=1.5)
fig = tc.pl.scatter(tdatas,
                    keys='$log_{10}(UMI)$',
                    cmap='Spectral_r',
                    position_key=['x','y'],
                    joint=True, point_size=4, 
                    ax=axs, 
                    noticks=False,
                    legend=True,
                    rasterized=True, 
                    cmap_vmin_vmax=[pucks.obs['$log_{10}(UMI)$'].min(),pucks.obs['$log_{10}(UMI)$'].max()])
plt.savefig(f"{fig_dir}FigureS11C.pdf", bbox_inches='tight')
plt.close()

## Proportion barplot

In [ ]:
multi_center=50
puck_ids = ['BC102','BC084','BC101','BC093','BC094','BC073','BC071','BC066','BC098','BC096']

In [ ]:
pucks_list = []

for i, puck_id in enumerate(puck_ids):
     file_path = f"{save_dir}puck_adata_mc{multi_center}_{puck_id}.h5ad"
     puck = sc.read(file_path)
     pucks_list.append(puck)

pucks = sc.concat(pucks_list, label='puck', index_unique='-')

In [ ]:
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

lineage_colors = {'CD4_Tconv': '#2452a3', 'Treg':'#f5be1c' , 'CD8_T': '#4ac0e3', 'NK_ILC': '#674498', 
                  'MoMac': '#44b143', 'DC': '#54a266', 'Mast': '#b5b5b6', 'B': '#da1817', 'Plasma': '#ee791d', 
                  'MSC':'#3c3c3c' , 'Endothelial': '#d58252', 'Epithelial': '#994923'}
cmap = ListedColormap(list(lineage_colors.values()))

In [ ]:
plot_composition_simple2(pucks, 'sample', 'predictions_lineage', 
                        x_order=puck_ids, 
                        y_order=['CD4_Tconv', 'Treg', 'CD8_T', 'NK_ILC', 'MoMac', 'DC', 'Mast', 'B',
                                 'Plasma', 'MSC', 'Endothelial', 'Epithelial'], 
                        norm=True, style='bar', 
                        bar_figsize=(8, 6), 
                        cmap=cmap, 
                        bar_width=0.6, bar_linkage=True, bar_link_alpha=0.5, 
                        heat_figsize=(5, 5), only_size=False, heat_size_scale=50, heat_marker='.', 
                        return_df=True, melt=False, return_df_annot=None)
plt.savefig(f"{fig_dir}FigureS11E.pdf", bbox_inches='tight')

## Find spatially contiguous regions in post-BCG bladder mucosa

In [ ]:
multi_center=50
puck_ids = ['BC096','BC098']

In [ ]:
pucks_list = []

for i, puck_id in enumerate(puck_ids):
     file_path = f"{save_dir}puck_adata_mc{multi_center}_{puck_id}.h5ad"
     puck = sc.read(file_path)
     pucks_list.append(puck)

pucks = sc.concat(pucks_list, label='puck', index_unique='-')

In [ ]:
position_weight = 0.9
resolution = 0.4

tc.tl.find_regions(adata = pucks,
                   key_added ='regions',
                   batch_key = "sample",
                   position_weight=position_weight,
                   resolution=resolution,
                   annotation_connectivity=None,
                   annotation_key='predictions',
                   position_connectivity=None,
                   position_key= ['x','y'],
                   n_neighbors = 15
                   )
pucks.obs['regions'] = pucks.obs['regions'].astype('category')

In [ ]:
rename_dict = {
    "0": "Region 1",
    "2": "Region 2",
    "1": "Region 3",
    "3": "Region 4"
}

pucks.obs['regions'] = pucks.obs['regions'].astype(str).map(rename_dict).astype("category")

region_colors = {
    'Region 1': '#023EFF',
    'Region 2': '#ff7c00',
    'Region 3': '#1ac938',
    'Region 4': '#e8000b',
}

In [ ]:
# Spatial plot (Fig.6D)
tc.pl.scatter(
    pucks,
    group_key='sample',
    colors=region_colors, 
    keys='regions',
    position_key=['x', 'y'],
    joint=True,
    point_size=1,
    axsize=[2, 2],
    noticks=False,
    axes_labels=['X', 'Y'],
    rasterized=True
)
plt.savefig(f"{fig_dir}Figure6D_spatial.pdf", bbox_inches='tight')
plt.close()

### GASTON restricted region

In [ ]:
puck = pucks[pucks.obs['sample'] == 'BC096'].copy()

coords_tmp_df = pd.read_csv("Reproducibility/Results/Slide-seqV2/GASTON/BC096/coords_mat_restrict.txt", sep='\t', index_col=0)
coords_tmp_df.columns = ['x', 'y']

puck_spatial_df = pd.DataFrame(index=puck.obs_names)
puck_spatial_df['x'] = puck.obs['x'].values
puck_spatial_df['y'] = puck.obs['y'].values

# Find rows in puck_spatial_df that approximately match coords_tmp_df
tolerance = 1e-3  # Adjust if necessary
matching_indices = puck_spatial_df[
    puck_spatial_df.apply(lambda row: np.any(
        np.isclose(coords_tmp_df["x"], row["x"], atol=tolerance) &
        np.isclose(coords_tmp_df["y"], row["y"], atol=tolerance)
    ), axis=1)
].index

# Subset the dataframe using the inferred index
puck_spatial_subset = puck_spatial_df.loc[matching_indices]
remove = np.setdiff1d(puck.obs_names,puck_spatial_subset.index)

# Create a new column in puck.obs to differentiate selected vs. unselected points
puck.obs['plot_category'] = puck.obs['regions'].astype('category')
puck.obs['plot_category'] = puck.obs['plot_category'].cat.add_categories(['Region_5'])
puck.obs.loc[remove, 'plot_category'] = 'Region_5'

# Define colors: all non-selected points will be grey
region_colors = {
    'Region 1': '#023EFF',
    'Region 2': '#ff7c00',
    'Region 3': '#1ac938',
    'Region 4': '#e8000b',
    'Region_5': 'whitesmoke'
}

# Spatial scatter plot
tc.pl.scatter(
    puck,
    keys='plot_category',
    position_key=['x', 'y'],
    colors=region_colors,  # Ensures non-selected points are grey
    joint=True,
    point_size=1,
    axsize=[2, 2],
    noticks=False,
    axes_labels=['X', 'Y'],
    rasterized=True
)
plt.savefig(f'Reproducibility/Results/Plots/Slide-seqV2/FigureS11G_region.pdf', bbox_inches='tight')
plt.close()
